In [3]:
"""
collect_data.py
===============
ETL Step 1: Download raw market data from Yahoo Finance and FRED.
Output: raw_gold_data.csv
"""

import datetime

import pandas as pd
import yfinance as yf
from fredapi import Fred

# ── Config ────────────────────────────────────────────────────────────────────

FRED_API_KEY = "308c46222636bb6d8743d4b4902a42d1"
START_DATE   = "2000-01-01"
END_DATE     = datetime.datetime.now().strftime("%Y-%m-%d")
OUTPUT_FILE  = "raw_gold_data.csv"

# ── Yahoo Finance tickers ─────────────────────────────────────────────────────
#
# daily_prices   → GC=F, SI=F, DX-Y.NYB, ^GSPC, ^VIX, EURUSD=X, USDCNY=X,
#                  RUB=X, PL=F, PA=F
# etf_flows      → GLD, IAU
# energy_mining  → CL=F, BZ=F, HG=F

YF_TICKERS: dict[str, str] = {
    # Precious metals
    "GC=F":     "Gold_Price",
    "SI=F":     "Silver_Price",
    "PL=F":     "Platinum_Price",
    "PA=F":     "Palladium_Price",
    # Equity indices
    "DX-Y.NYB": "DXY_Index",
    "^GSPC":    "SP500",
    "^VIX":     "VIX_Index",
    # FX rates (USD base)
    "EURUSD=X": "USD_EUR",
    "USDCNY=X": "USD_CNY",
    "RUB=X":    "USD_RUB",
    # Gold ETFs
    "GLD":      "GLD_Price",
    "IAU":      "IAU_Price",
    # Energy & industrial metals → energy_mining table
    "CL=F":     "WTI_Oil",
    "BZ=F":     "Brent_Oil",
    "HG=F":     "Copper_Price",
}

# ── FRED series ───────────────────────────────────────────────────────────────
#
# All land in macro_indicators.
# PPIACO is also used in transform_data.py to build Mining_Cost_Index.

FRED_SERIES: dict[str, str] = {
    "FEDFUNDS": "Fed_Rate",       # Federal funds rate
    "DGS10":    "Treasury_10Y",   # 10-year UST yield
    "DGS2":     "Treasury_2Y",    # 2-year UST yield
    "CPIAUCSL": "CPI_Index",      # Consumer price index
    "UNRATE":   "Unemployment",   # Unemployment rate
    "M2SL":     "M2_Supply",      # M2 money supply
    "PPIACO":   "PPI",            # Producer price index (needed for mining_cost_index)
}

def download_yfinance() -> pd.DataFrame:
    """Download close prices for all YF_TICKERS. Returns a wide DataFrame indexed by Date."""
    print(f"  Downloading {len(YF_TICKERS)} tickers from Yahoo Finance...")

    raw = yf.download(
        list(YF_TICKERS.keys()),
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
    )

    # yfinance returns a MultiIndex (field, ticker) when multiple tickers are requested
    if isinstance(raw.columns, pd.MultiIndex):
        df = raw["Close"].copy()
    else:
        df = raw.copy()

    # Strip stray whitespace/newlines from column names (yfinance quirk)
    df.columns = pd.Index([str(c).strip() for c in df.columns])
    df = df.rename(columns=YF_TICKERS)

    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"

    # Brent history starts ~2006; backfill early gaps with WTI as proxy
    if "Brent_Oil" in df.columns and "WTI_Oil" in df.columns:
        df["Brent_Oil"] = df["Brent_Oil"].fillna(df["WTI_Oil"])

    print(f"  yfinance: {len(df)} rows, {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    return df

yf_df   = download_yfinance()
yf_df

[*********************100%***********************]  15 of 15 completed

  yfinance: 6838 rows, 15 columns
  Columns: ['Brent_Oil', 'WTI_Oil', 'DXY_Index', 'USD_EUR', 'Gold_Price', 'GLD_Price', 'Copper_Price', 'IAU_Price', 'Palladium_Price', 'Platinum_Price', 'USD_RUB', 'Silver_Price', 'USD_CNY', 'SP500', 'VIX_Index']


,Brent_Oil,WTI_Oil,DXY_Index,USD_EUR,Gold_Price,GLD_Price,Copper_Price,IAU_Price,Palladium_Price,Platinum_Price,USD_RUB,Silver_Price,USD_CNY,SP500,VIX_Index
Date,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,100.220001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1455.219971,24.209999
2000-01-04,NaN,NaN,100.410004,NaN,NaN,NaN,NaN,NaN,441.899994,429.700012,NaN,NaN,NaN,1399.420044,27.010000
2000-01-05,NaN,NaN,100.379997,NaN,NaN,NaN,NaN,NaN,438.100006,419.899994,NaN,NaN,NaN,1402.109985,26.410000
2000-01-06,NaN,NaN,100.650002,NaN,NaN,NaN,NaN,NaN,435.299988,412.000000,NaN,NaN,NaN,1403.449951,25.730000
2000-01-07,NaN,NaN,100.800003,NaN,NaN,NaN,NaN,NaN,443.899994,414.000000,NaN,NaN,NaN,1441.469971,21.719999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-09,95.919998,97.870003,98.820000,1.165936,4792.200195,437.910004,5.7480,89.720001,1553.000000,2096.600098,78.548210,76.277000,6.8287,6824.660156,19.490000
2026-04-10,95.199997,96.570000,98.650002,1.169126,4761.899902,437.130005,5.8705,89.559998,1528.099976,2049.100098,77.619156,76.323997,6.8314,6816.890137,19.230000
2026-04-13,99.360001,99.080002,98.370003,1.167515,4742.399902,435.359985,5.9760,89.209999,1569.800049,2061.800049,76.548508,75.523003,6.8278,6886.240234,19.120001


In [4]:
def download_fred() -> pd.DataFrame:
    """Download all FRED_SERIES. Returns a wide DataFrame indexed by Date."""
    print(f"  Downloading {len(FRED_SERIES)} series from FRED...")

    fred = Fred(api_key=FRED_API_KEY)
    frames: dict[str, pd.Series] = {}

    for code, col_name in FRED_SERIES.items():
        try:
            s = fred.get_series(code, observation_start=START_DATE)
            s.index = pd.to_datetime(s.index).tz_localize(None)
            frames[col_name] = s
            print(f"    OK {code} -> {col_name} ({len(s)} obs)")
        except Exception as exc:
            print(f"    WARN {code}: {exc}")

    df = pd.DataFrame(frames)
    df.index.name = "Date"
    return df


fred_df = download_fred()
fred_df

    OK FEDFUNDS -> Fed_Rate (315 obs)
    OK DGS10 -> Treasury_10Y (6857 obs)
    OK DGS2 -> Treasury_2Y (6857 obs)
    OK CPIAUCSL -> CPI_Index (315 obs)
    OK UNRATE -> Unemployment (315 obs)
    OK M2SL -> M2_Supply (314 obs)
    OK PPIACO -> PPI (315 obs)


,Fed_Rate,Treasury_10Y,Treasury_2Y,CPI_Index,Unemployment,M2_Supply,PPI
Date,,,,,,,
2000-01-01,5.45,NaN,NaN,169.3,4.0,4667.6,128.3
2000-01-03,NaN,6.58,6.38,NaN,NaN,NaN,NaN
2000-01-04,NaN,6.49,6.30,NaN,NaN,NaN,NaN
2000-01-05,NaN,6.62,6.38,NaN,NaN,NaN,NaN
2000-01-06,NaN,6.57,6.35,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2026-04-08,NaN,4.29,3.79,NaN,NaN,NaN,NaN
2026-04-09,NaN,4.29,3.78,NaN,NaN,NaN,NaN
2026-04-10,NaN,4.31,3.81,NaN,NaN,NaN,NaN


In [5]:
def collect_data() -> None:
    """Run full Step 1: download yfinance + FRED, merge, save to CSV."""
    print(f"=== collect_data: {START_DATE} -> {END_DATE} ===\n")

    yf_df   = download_yfinance()
    fred_df = download_fred()

    # Outer join on date index.
    # FRED data is lower-frequency (monthly/quarterly) — NaNs filled later
    # in transform_data.py via ffill. Keep as-is here.
    df = pd.concat([yf_df, fred_df], axis=1).sort_index()

    df.to_csv(OUTPUT_FILE)
    print(f"\nSaved -> {OUTPUT_FILE}")
    print(f"  Rows:    {len(df)}")
    print(f"  Columns: {df.shape[1]}")
    print(f"  Period:  {df.index.min().date()} -> {df.index.max().date()}")


collect_data()

=== collect_data: 2000-01-01 -> 2026-04-16 ===



[*********************100%***********************]  15 of 15 completed


  yfinance: 6838 rows, 15 columns
  Columns: ['Brent_Oil', 'WTI_Oil', 'DXY_Index', 'USD_EUR', 'Gold_Price', 'GLD_Price', 'Copper_Price', 'IAU_Price', 'Palladium_Price', 'Platinum_Price', 'USD_RUB', 'Silver_Price', 'USD_CNY', 'SP500', 'VIX_Index']
    OK FEDFUNDS -> Fed_Rate (315 obs)
    OK DGS10 -> Treasury_10Y (6857 obs)
    OK DGS2 -> Treasury_2Y (6857 obs)
    OK CPIAUCSL -> CPI_Index (315 obs)
    OK UNRATE -> Unemployment (315 obs)
    OK M2SL -> M2_Supply (314 obs)
    OK PPIACO -> PPI (315 obs)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17932\581513918.py:11: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([yf_df, fred_df], axis=1).sort_index()



Saved -> raw_gold_data.csv
  Rows:    6950
  Columns: 22
  Period:  2000-01-01 -> 2026-04-15
